# 1. Fixed split and preprocessing

Use the saved split and class mapping; this notebook never recreates them. Exact decoded-image groups were sorted and split with class-stratified `train_test_split(test_size=0.20, random_state=42)`. Keeping duplicate groups together avoids identified exact-duplicate leakage. Group stratification gives approximately 80/20 at image level. Unknown related crops or animals may still overlap.

Inputs: `data/processed/` metadata. Outputs: the original split evidence and preprocessing/augmentation figures in `outputs/`; optional examples below are displayed only.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch

# Set this explicitly on Kaggle to the extracted project folder.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                         if (p / "src/models.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the extracted project containing src/ and data/processed/")
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.project import RUNS, load_metadata, read_strings, find_image_directory, find_csv
DATA_ROOT = (Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2")
             if Path("/kaggle").exists() else PROJECT_ROOT / "data/raw")
OUTPUT_ROOT = (Path("/kaggle/working/reproduction") if Path("/kaggle").exists()
               else PROJECT_ROOT / "outputs/reproduction")
TRAIN_IMAGE_DIRECTORY = None  # Optional existing cache or explicit train/train directory.
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
RUN_TRAINING = False
RUN_TEST_PREDICTION = False
RUN_EVALUATION = False
split, mapping = load_metadata(PROJECT_ROOT)
print("Project:", PROJECT_ROOT, "Device:", DEVICE)
print(split["split"].value_counts().to_dict())

In [ ]:
from src.data_pipeline import reference_preprocess, FacialFeatureDataset, ImageNetNormalizedDataset, seed_everything
from IPython.display import Image, display
RUN_EXAMPLES = False
counts = pd.crosstab(split["class"], split["split"])
counts["validation_fraction"] = counts["validation"] / counts[["train", "validation"]].sum(axis=1)
display(counts)
display(pd.Series(mapping["class_to_index"], name="index"))
assert all(mapping["index_to_class"][str(i)] == name for name, i in mapping["class_to_index"].items())
print("All eight classes retained; IDs, byte hashes and pixel hashes are partition-disjoint.")

## 2. Geometry and augmentation

Convert to RGB, resize the longest side to 224 with bilinear interpolation and integer half-up rounding, then centre-pad with RGB(128,128,128). An odd padding pixel goes on the right or bottom. Convert once to a float32 CHW tensor and divide by 255. Images are neither cropped nor stretched.

Training alone applies horizontal flip (p=0.5) and brightness/contrast jitter (0.1 each) before resizing, keeping padding grey. E1/E2 apply ImageNet mean/std normalisation once afterward; E0 does not. Validation and test transformations are deterministic. Raw files are unchanged.

In [ ]:
for name in ["preprocessing_preview.png", "training_augmentation_preview.png", "validation_split_overview.png"]:
    display(Image(filename=str(PROJECT_ROOT / "outputs/figures" / name), width=850))

In [ ]:
if RUN_EXAMPLES:
    directory = TRAIN_IMAGE_DIRECTORY or find_image_directory(DATA_ROOT, split["image_id"])
    seed_everything(42)
    training = FacialFeatureDataset(PROJECT_ROOT, "train", directory)
    validation = FacialFeatureDataset(PROJECT_ROOT, "validation", directory)
    image, label, image_id = validation[0]
    repeated = validation[0][0]
    assert image.shape == (3, 224, 224) and image.dtype == torch.float32
    assert torch.equal(image, repeated) and bool(torch.isfinite(image).all())
    assert 0 <= image.min() <= image.max() <= 1
    print(image_id, label, tuple(image.shape))
else:
    print("Optional image examples disabled.")